# Chapter 9 assignment: diagnosing and repairing PINN training failures

Companion notebook for Chapter 9, *Training Difficulties, Diagnostics and Remedies*.
Everything runs on a CPU in double precision; each training run takes well under a minute.

**Learning goals.** After this assignment you can

1. measure spectral bias with a frequency-resolved error and remove it with Fourier features;
2. compute per-term gradient norms and implement learning-rate annealing for loss weights;
3. predict the loss of a spurious near-equilibrium solution and remove it with a relative residual;
4. implement causal weights for a long time interval;
5. use a validation residual and residual-based adaptive refinement (RAR) as diagnostics.

Run the notebook from the repository root (or set `PYTHONPATH` to it) so that `import pinnode` works.
Cells marked `# TODO` must be completed; the *check your answer* cells then run without errors.

In [ ]:
import math, sys, os
import numpy as np
import torch
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))   # repository root when run from notebooks/
from pinnode import (MLP, FourierFeatureNet, ConstrainedModel, derivative, to_tensor, train,
                     rel_l2, linf, configure_torch, set_seed, dirichlet_transform)
configure_torch("float64", threads=1)
set_seed(1234)
print("torch", torch.__version__)

## Theory recap

* **Training dynamics.** For gradient flow on $\frac{1}{2M}\|\boldsymbol r\|^2$ the residual vector obeys
  $\dot{\boldsymbol r} = -\frac1M K\boldsymbol r$ with the neural tangent kernel $K = JJ^\top$, $J=\partial\boldsymbol r/\partial\theta$.
  Components along eigenvectors with small eigenvalues decay slowly; for standard MLPs these are the oscillatory ones (*spectral bias*).
* **Imbalance.** For $y''=f$ with a solution oscillating at frequency $\omega$, $\|\nabla_\theta L_r\|$ grows like $\omega^4$ while
  $\|\nabla_\theta L_b\| = O(1)$. Learning-rate annealing sets $\hat\lambda_b = \max_k|\partial_{\theta_k}L_r| / \mathrm{mean}_k|\partial_{\theta_k}L_b|$ and
  $\lambda_b \leftarrow (1-\alpha)\lambda_b + \alpha\hat\lambda_b$.
* **Equilibria.** For the logistic equation $y'=ry(1-y)$ the decaying family $a e^{-rt}$ has soft-constraint loss
  $\lambda_{ic}(a-y_0)^2 + 2ra^2/T$, minimal value $L^*(T) = 2r\lambda_{ic}y_0^2/(\lambda_{ic}T+2r)$.
* **Causality.** With residuals grouped into consecutive chunks $L_1,\dots,L_{N}$, the weights $w_i=\exp(-\varepsilon\sum_{k<i}L_k)$
  (no gradient through $w$) switch on later chunks only when earlier ones are fitted.
* **A posteriori bound.** If $f$ is one-sided Lipschitz with constant $\mu$, $\|e(t)\|\le e^{\mu t}\|e(0)\| + \int_0^t e^{\mu(t-s)}\|r(s)\|\,ds$.

## Task 1: spectral bias and Fourier features

Problem: $y'(t) = 2\pi\cos(2\pi t) + 0.1\cdot 2\pi \nu\cos(2\pi \nu t)$, $y(0)=0$ on $[0,1]$ with $\nu=8$;
exact solution $y=\sin(2\pi t)+0.1\sin(2\pi \nu t)$. Use the hard initial condition $y_\theta = t\,N_\theta(t)$.

1. Complete `amplitude(e, t, k)`, the amplitude of the mode with $k$ cycles per unit time in the error $e(t)$
   (use $a_k = 2\int_0^1 e\cos(2\pi kt)\,dt$, $b_k = 2\int_0^1 e\sin(2\pi kt)\,dt$, amplitude $\sqrt{a_k^2+b_k^2}$).
2. Train a tanh MLP and a Fourier-feature network (choose $\sigma$ with the rule $\sigma\approx kT/2$ of Section 9.2) for 3000 Adam steps
   and record $A_1$ and $A_\nu$ every 100 steps. Which mode is learned first by the MLP?

In [ ]:
NU = 8
t_col = to_tensor(np.linspace(0, 1, 160), requires_grad=True)
t_test = np.linspace(0, 1, 1001)
y_exact = np.sin(2*np.pi*t_test) + 0.1*np.sin(2*np.pi*NU*t_test)
f = lambda t: 2*np.pi*torch.cos(2*np.pi*t) + 0.1*2*np.pi*NU*torch.cos(2*np.pi*NU*t)

def amplitude(e, t, k):
    # TODO: return the amplitude of the mode exp(2 pi i k t) in e(t) on [0, 1] (trapezoidal rule)
    raise NotImplementedError

def train_net(net, n_adam=3000):
    model = ConstrainedModel(net, lambda t, n: t * n)
    amps = {"it": [], "A1": [], "Anu": []}
    def loss_fn():
        r = derivative(model(t_col), t_col) - f(t_col.detach())
        L = (r**2).mean(); return L, {"residual": L}
    def eval_fn():
        with torch.no_grad():
            e = model(to_tensor(t_test)).numpy().ravel() - y_exact
        amps["A1"].append(amplitude(e, t_test, 1)); amps["Anu"].append(amplitude(e, t_test, NU))
        return {"rel_l2": float(np.linalg.norm(e) / np.linalg.norm(y_exact))}
    hist = train(net.parameters(), loss_fn, adam_iters=n_adam, lr=1e-3, lbfgs_iters=0,
                 eval_fn=eval_fn, eval_every=100, verbose=False)
    amps["it"] = hist.eval_iters
    return model, hist, amps

torch.manual_seed(1234)
mlp = MLP(1, 1, width=48, depth=3, activation="tanh", domain=(0, 1))
sigma = None   # TODO: choose sigma for the Fourier-feature network
torch.manual_seed(1234)
ff = FourierFeatureNet(1, 1, n_features=32, sigma=sigma, width=48, depth=3, domain=(0, 1))
model_mlp, hist_mlp, amps_mlp = train_net(mlp)
model_ff, hist_ff, amps_ff = train_net(ff)
print("final rel. L2: MLP %.2e, Fourier features %.2e" % (hist_mlp.final_metric("rel_l2"), hist_ff.final_metric("rel_l2")))

**Check your answer.**

In [ ]:
tt = np.linspace(0, 1, 2001)
assert abs(amplitude(0.3*np.sin(2*np.pi*5*tt + 0.4), tt, 5) - 0.3) < 1e-3, "amplitude() is not correct"
assert amps_ff["Anu"][-1] < amps_mlp["Anu"][-1], "the Fourier-feature network should resolve the fast mode better"
print("Task 1 passed. MLP: A_1 = %.1e, A_nu = %.1e;  FF: A_1 = %.1e, A_nu = %.1e"
      % (amps_mlp["A1"][-1], amps_mlp["Anu"][-1], amps_ff["A1"][-1], amps_ff["Anu"][-1]))

## Task 2: gradient norms and learning-rate annealing

Problem: $y'' = -\omega^2\sin(\omega t)$, $y(0)=0$, $y(1)=1$, $\omega=4\pi$ (exact $y=\sin(\omega t)+t$), soft boundary conditions.

1. Complete `grad_norm(loss, params)` (Euclidean norm of the gradient). Use
   `torch.autograd.grad(loss, params, retain_graph=True, allow_unused=True)`: parameters that do not influence a
   term, such as the output bias for a residual that contains only $y''$, give `None` and contribute zero.
2. Print $\|\nabla L_r\|$ and $\|\nabla L_b\|$ at initialization. Compare their ratio with the $\omega^4$ estimate.
3. Complete the annealing update in `weight_update` and compare the boundary error after 3000 Adam steps with $\lambda_b=1$.

In [ ]:
omega = 4*math.pi
t_c = to_tensor(np.linspace(0, 1, 100), requires_grad=True)
t_b, y_b = to_tensor([0.0, 1.0]), to_tensor([0.0, 1.0])

def grad_norm(loss, params):
    # TODO: return ||d loss / d params||_2 (keep the graph: retain_graph=True; allow_unused=True)
    raise NotImplementedError

def run_bvp(anneal, n_adam=3000, alpha=0.9):
    torch.manual_seed(1234)
    net = MLP(1, 1, 32, 3, "tanh", domain=(0, 1)); params = list(net.parameters())
    state = {"lam_b": 1.0}
    def terms():
        y = net(t_c); r = derivative(y, t_c, 2) + omega**2*torch.sin(omega*t_c)
        return (r**2).mean(), ((net(t_b) - y_b)**2).mean()
    def loss_fn():
        Lr, Lb = terms(); return Lr + state["lam_b"]*Lb, {"residual": Lr, "boundary": Lb}
    def weight_update(k):
        if anneal and k % 10 == 0:
            Lr, Lb = terms()
            # TODO: compute lam_hat = max|grad L_r| / mean|grad L_b| (flattened gradients, unused -> 0)
            #       and update state["lam_b"]
            pass
    Lr, Lb = terms()
    g0 = (grad_norm(Lr, params), grad_norm(Lb, params))
    train(params, loss_fn, adam_iters=n_adam, lr=1e-3, lbfgs_iters=0, weight_update=weight_update, verbose=False)
    with torch.no_grad():
        bc_err = float((net(t_b) - y_b).abs().max())
        e = rel_l2(net(to_tensor(np.linspace(0, 1, 1001))), np.sin(omega*np.linspace(0, 1, 1001)) + np.linspace(0, 1, 1001))
    return g0, bc_err, e, state["lam_b"]

g0, bc_fixed, e_fixed, _ = run_bvp(anneal=False)
_, bc_lra, e_lra, lam_final = run_bvp(anneal=True)
print("gradient norms at initialization: ||grad L_r|| = %.2e, ||grad L_b|| = %.2e" % g0)
print("fixed weights: BC error %.2e, rel. L2 %.2e" % (bc_fixed, e_fixed))
print("annealing:     BC error %.2e, rel. L2 %.2e, final lambda_b = %.2e" % (bc_lra, e_lra, lam_final))

**Check your answer.**

In [ ]:
assert g0[0] > 10*g0[1], "the residual gradient should dominate at initialization"
assert lam_final > 10, "the annealed boundary weight should grow well above 1"
assert bc_lra < bc_fixed, "annealing should reduce the boundary error"
print("Task 2 passed.")

## Task 3: the attraction of an equilibrium

Logistic equation $y'=ry(1-y)$, $r=1$, $y_0=0.1$ on $[0,T]$ with $T=10$.

1. Complete `spurious_loss(T, lam_ic)` from the formula in the recap and evaluate it for $\lambda_{ic}=1$.
2. Train with a soft initial condition and compare the final loss with $L^*(T)$. Plot or print $y_\theta(T)$.
3. Implement the relative residual: trial function $y_\theta = y_0\exp(tN_\theta(t))$ and residual
   $(\ln y_\theta)' - r(1-y_\theta) = N_\theta + tN_\theta' - r(1-y_\theta)$.

In [ ]:
r_, y0, T = 1.0, 0.1, 10.0
t_l = to_tensor(np.linspace(0, T, 128), requires_grad=True)
t_lt = np.linspace(0, T, 1001); y_log = y0*np.exp(r_*t_lt)/(1 + y0*(np.exp(r_*t_lt) - 1))

def spurious_loss(T, lam_ic=1.0):
    # TODO: return 2 r lam y0^2 / (lam T + 2 r)
    raise NotImplementedError

def run_logistic(relative, n_adam=3000):
    torch.manual_seed(1234)
    net = MLP(1, 1, 32, 3, "tanh", domain=(0, T))
    if relative:
        model = ConstrainedModel(net, lambda t, n: y0*torch.exp(t*n))
    else:
        model = net
    def loss_fn():
        y = model(t_l)
        if relative:
            # TODO: relative residual N + t N' - r (1 - y)
            raise NotImplementedError
        res = derivative(y, t_l) - r_*y*(1 - y)
        Lr = (res**2).mean(); Lic = ((model(to_tensor([0.0])) - y0)**2).mean()
        return Lr + Lic, {"residual": Lr, "ic": Lic}
    h = train(net.parameters(), loss_fn, adam_iters=n_adam, lr=1e-3, lbfgs_iters=200, verbose=False)
    with torch.no_grad():
        yp = model(to_tensor(t_lt)).numpy().ravel()
    return h.final_loss, rel_l2(yp, y_log), yp[-1]

L_soft, e_soft, yT_soft = run_logistic(relative=False)
L_rel, e_rel, yT_rel = run_logistic(relative=True)
print("predicted spurious loss L*(10) = %.2e" % spurious_loss(T))
print("soft IC:           final loss %.2e, rel. L2 %.2e, y(T) = %.3f" % (L_soft, e_soft, yT_soft))
print("relative residual: final loss %.2e, rel. L2 %.2e, y(T) = %.3f" % (L_rel, e_rel, yT_rel))

**Check your answer.**

In [ ]:
assert abs(spurious_loss(10.0) - 2*0.01/12) < 1e-12
assert e_rel < 1e-2, "the relative residual should recover the logistic solution"
print("Task 3 passed. Soft IC %s." % ("collapsed to the spurious solution" if e_soft > 0.1 else "converged in this run"))

## Task 4: causal weights

Harmonic oscillator $y''+y=0$, $y(0)=1$, $y'(0)=0$ on $[0, 8\pi]$ with the hard initial conditions $y_\theta=1+t^2N_\theta(t)$.

1. Complete `causal_weights(L_chunks, eps)`: $w_1=1$, $w_i=\exp(-\varepsilon\sum_{k<i}L_k)$, computed without gradient.
2. Train with $\varepsilon$ increased through $10^{-2},10^{-1},1,10,100$ whenever $\min_i w_i>0.99$, then compare with the unweighted loss.

In [ ]:
T4, NC = 8*math.pi, 32
t_h = to_tensor(np.linspace(0, T4, 256), requires_grad=True)
t_ht = np.linspace(0, T4, 1001)

def causal_weights(L_chunks, eps):
    # TODO: w_1 = 1, w_i = exp(-eps * sum_{k<i} L_k), inside torch.no_grad()
    raise NotImplementedError

def run_oscillator(causal, n_adam=4000):
    torch.manual_seed(1234)
    net = MLP(1, 1, 64, 3, "tanh", domain=(0, T4))
    model = ConstrainedModel(net, lambda t, n: 1.0 + t**2*n)
    st = {"eps_list": [1e-2, 1e-1, 1.0, 10.0, 100.0], "k": 0}
    def loss_fn():
        y = model(t_h); r = derivative(y, t_h, 2) + y
        if not causal:
            L = (r**2).mean(); return L, {"residual": L}
        Lc = (r**2).reshape(NC, -1).mean(dim=1)
        w = causal_weights(Lc, st["eps_list"][st["k"]])
        if float(w.min()) > 0.99 and st["k"] < 4:
            st["k"] += 1
        return (w*Lc).mean(), {"residual": (r**2).mean(), "w_min": float(w.min())}
    train(net.parameters(), loss_fn, adam_iters=n_adam, lr=1e-3, lbfgs_iters=0, verbose=False)
    with torch.no_grad():
        return rel_l2(model(to_tensor(t_ht)), np.cos(t_ht)), st["eps_list"][st["k"]]

e_van, _ = run_oscillator(False)
e_cau, eps_reached = run_oscillator(True)
print("rel. L2: unweighted %.2e, causal %.2e (eps reached %g)" % (e_van, e_cau, eps_reached))

**Check your answer.**

In [ ]:
w = causal_weights(torch.tensor([0.5, 0.1, 0.2, 0.3]), 1.0)
assert torch.allclose(w, torch.exp(-torch.tensor([0.0, 0.5, 0.6, 0.8]))), "causal_weights() is not correct"
assert not w.requires_grad
print("Task 4 passed. Compare the two errors with Section 9.6: how far did eps get, and why?")

## Task 5: validation residual and residual-based adaptive refinement

1. For the Fourier-feature network of Task 1, retrain with only 24 collocation points and $\sigma=20$.
   Compute the mean squared residual on the training points and on 1001 fresh points (`validation_residual`).
2. Implement one RAR step for the stiff problem $y'=-100(y-\cos t)$, $y(0)=0$ on $[0,2\pi]$:
   evaluate the residual on 4000 uniform candidates and add the 5 candidates with the largest $|r|$ that are not
   already collocation points.
   Where do the new points lie, and why?

In [ ]:
def validation_residual(model, t_eval, rhs):
    # TODO: mean squared residual y' - rhs(t) of the model on the points t_eval (numpy array)
    raise NotImplementedError

def rar_step(model, points, residual_fn, n_cand=4000, n_add=5, T=2*math.pi):
    # TODO: evaluate |residual_fn(model, t)| on n_cand uniform candidates in [0, T], exclude candidates that
    #       are already in points, and return np.sort(np.concatenate([points, the n_add with largest |r|]))
    raise NotImplementedError

lam = 100.0
def stiff_residual(model, t):
    y = model(t); return (derivative(y, t) + lam*(y - torch.cos(t))) / lam

torch.manual_seed(1234)
stiff_net = MLP(1, 1, 32, 3, "tanh", domain=(0, 2*math.pi))
stiff_model = ConstrainedModel(stiff_net, lambda t, n: t*n)
pts = np.linspace(0, 2*math.pi, 20)
t_s = to_tensor(pts, requires_grad=True)
train(stiff_net.parameters(), lambda: ((stiff_residual(stiff_model, t_s)**2).mean(), {}), adam_iters=1000,
      lr=1e-3, lbfgs_iters=0, verbose=False)
new_pts = rar_step(stiff_model, pts, stiff_residual)
print("added points:", np.setdiff1d(new_pts, pts))

**Check your answer.**

In [ ]:
added = np.setdiff1d(new_pts, pts)
assert len(added) == 5
print("Task 5: %d of 5 new points lie in the initial layer t < 0.05." % np.sum(added < 0.05))

## Reflection

Write a short paragraph for each task: which diagnostic revealed the problem, what the remedy changed in the
mechanism (NTK spectrum, gradient balance, loss landscape, causality, sampling), and what it cost.
Compare every PINN result with a classical solver (`scipy.integrate.solve_ivp`) on the same problem.